# 00 · Warm-up: what does wrist EMG look like?

**10 minutes, everyone.** You'll load one recording from Meta's dataset and plot the
16 EMG channels with the gesture labels on top.

Surface EMG picks up the tiny electrical signals your forearm muscles make when they
fire. The wristband has 16 electrodes around the wrist and samples each one 2,000
times a second. When you press your index finger or swipe your thumb, a different mix
of channels lights up. That pattern is what the model learns.

*Data: Meta, generic-neuromotor-interface (CC BY-NC 4.0). Cite the Nature 2025 paper.*

In [ ]:
# --- Setup (run once) -------------------------------------------------------
# Local: you've already run `pip install -r requirements.txt` in this repo, so skip ahead.
# Colab: this cell clones the repo, installs it, and points at the shared data.
import os, sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/sandeepvijayarao09/wearable-ai-build-day.git"  # organizers fill in
    if not os.path.exists("/content/repo"):
        !git clone -q {REPO_URL} /content/repo
    %cd /content/repo
    !pip install -q -r requirements.txt
    # Shared data: in Drive, right-click the shared "emg_data" folder ->
    # Organize -> Add shortcut -> My Drive. Then:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["EMG_DATA_DIR"] = "/content/drive/MyDrive/emg_data"
    os.environ["EMG_MODELS_DIR"] = "/content/drive/MyDrive/emg_data/emg_models"
else:
    # Run from the repo root so `emg_kit` imports.
    if not os.path.exists("emg_kit") and os.path.exists("../emg_kit"):
        os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from emg_kit.core import DATA_DIR, GESTURES, load_recording, test_partitions

parts = test_partitions()
print("Test users available here:", list(parts))
user = list(parts)[0]
path, start, end = parts[user]
rec = load_recording(path, start, end, user)
print(f"{user}: {rec.emg.shape[0]} channels x {rec.emg.shape[1]:,} samples "
      f"= {rec.seconds/60:.1f} minutes, {len(rec.prompts)} labelled gestures")

The labels are just a table of *which gesture* happened *when*. Have a look:

In [ ]:
rec.prompts.head(10)

In [ ]:
rec.prompts["name"].value_counts()

## Plot 6 seconds

Each grey line is one electrode, stacked so they don't overlap. Black lines mark the
moments a gesture was performed.

In [ ]:
T0 = 60.0      # seconds into the recording -- try changing this
SPAN = 6.0     # seconds to show

t = rec.times - rec.times[0]
win = (t >= T0) & (t < T0 + SPAN)

fig, ax = plt.subplots(figsize=(13, 6))
spacing = 4 * np.std(rec.emg[:, win])
for ch in range(rec.emg.shape[0]):
    ax.plot(t[win], rec.emg[ch, win] + ch * spacing, lw=0.6, color="0.45")

labels = rec.prompts[(rec.prompts.time - rec.times[0]).between(T0, T0 + SPAN)]
for _, row in labels.iterrows():
    x = row.time - rec.times[0]
    ax.axvline(x, color="k", lw=1)
    ax.text(x, 16 * spacing, row["name"], rotation=90, va="bottom", ha="right", fontsize=9)

ax.set_yticks([i * spacing for i in range(16)], [f"ch{i}" for i in range(16)], fontsize=7)
ax.set_xlabel("time (s)")
ax.set_title(f"{user}: 16-channel wrist EMG with gesture labels")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()

**Try it:** change `T0` to find a `thumb_click`. Which channels move the most? Now find
an `index_press`. Is it the same channels?

That's the whole problem: different gestures, different patterns across 16 channels.
Next, the Data team runs Meta's model on this ->
`01_evaluate.ipynb`.